# Assignment 2: dashboard

Run with `uv run python main.py app`. Loads the committed `weights/best.pt` and
the outputs `train.ipynb` saved to `artifacts/`. Nothing is trained here.

In [ ]:
import json
from pathlib import Path

import gradio as gr
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from ultralytics import YOLO

import costing

ROOT = Path.cwd()
ARTIFACTS = ROOT / "artifacts"


def load_json(name):
    return json.loads((ARTIFACTS / name).read_text())


model = YOLO(str(ROOT / "weights" / "best.pt"))
config = load_json("config.json")
op = load_json("operating_point.json")
detection = load_json("detection_metrics.json")
det_cm = load_json("confusion_matrix.json")
dataset_stats = load_json("dataset_stats.json")
sweep = pd.read_csv(ARTIFACTS / "cost_sweep.csv")
test = pd.read_csv(ARTIFACTS / "scores_test.csv")

COSTS, OPS = config["costs"], config["ops"]
NAMES = config["dataset"]["classes"]
POSITIVE = op["positive_class"]
POS_ID = NAMES.index(POSITIVE)

## Live costing

In [ ]:
def band_view(low, high, base_rate):
    low, high = min(low, high), max(low, high)
    y, s = test["y_true"].to_numpy(), test["score"].to_numpy()
    outcome = np.where(s >= high, "fail", np.where(s < low, "pass", "review"))
    counts = pd.crosstab(pd.Series(np.where(y == 1, f"{POSITIVE} present", "no " + POSITIVE), name="truth"),
                         pd.Series(outcome, name="outcome")).reindex(columns=["pass", "review", "fail"], fill_value=0)
    counts = counts.reset_index()

    cost, share = costing.band_cost(s, y, low, high, COSTS, base_rate)
    alone, _ = costing.band_cost(s, y, op["threshold"], op["threshold"], COSTS, base_rate)
    alts = costing.alternatives(COSTS, base_rate)
    reviewers = costing.reviewers_needed(share, OPS)
    summary = (
        f"### Expected cost: €{cost:,.0f} per 1,000 cases\n"
        f"- Sent to review: **{share:.1%}**, about **{share * OPS['cases_per_day']:,.0f} cases/day**, "
        f"needing **{reviewers:.2f}** reviewers ({OPS['reviewers_available']} available)\n"
        f"- Base rate assumed: {base_rate:.1%}. Counts below are the raw test split; the cost re-weights them to the base rate."
    )
    compare = pd.DataFrame([
        {"policy": "Do nothing", "cost per 1,000 (€)": alts["do_nothing"]},
        {"policy": "Person checks everything", "cost per 1,000 (€)": alts["person_checks_everything"]},
        {"policy": f"Model alone (t = {op['threshold']:.2f})", "cost per 1,000 (€)": alone},
        {"policy": f"Model + review ({low:.2f} – {high:.2f})", "cost per 1,000 (€)": cost},
    ]).round(1)
    return summary, counts, compare, cost_curve(low, high, base_rate)


def cost_curve(low, high, base_rate):
    fig = go.Figure()
    if abs(base_rate - config["base_rate"]) < 1e-9:
        val_curve, test_curve, coco_curve = (sweep["cost_per_1000_val"], sweep["cost_per_1000_test"],
                                             sweep["cost_per_1000_coco_test"])
    else:  # re-price live when the base rate slider moves away from the saved assumption
        val_curve = None
        test_curve = costing.sweep(test["score"], test["y_true"], COSTS, base_rate)["cost_per_1000"]
        coco_curve = costing.sweep(test["score_coco"], test["y_true"], COSTS, base_rate)["cost_per_1000"]
    t = sweep["threshold"]
    if val_curve is not None:
        fig.add_trace(go.Scatter(x=t, y=val_curve, name="finetuned (val)", line=dict(dash="dot")))
    fig.add_trace(go.Scatter(x=t, y=test_curve, name="finetuned (test)"))
    fig.add_trace(go.Scatter(x=t, y=coco_curve, name="COCO zero-shot (test)", line=dict(color="grey")))
    fig.add_vline(x=op["threshold"], line_color="red", annotation_text=f"chosen {op['threshold']:.2f}")
    fig.add_vrect(x0=min(low, high), x1=max(low, high), fillcolor="orange", opacity=0.15, line_width=0,
                  annotation_text="review band")
    fig.update_layout(xaxis_title="confidence threshold", yaxis_title="expected cost per 1,000 cases (€)",
                      height=420, margin=dict(t=30))
    return fig

## Predictions on an uploaded image

In [ ]:
def predict_image(image, low, high):
    if image is None:
        return None, "Upload an image, or pick a test-split image below."
    low, high = min(low, high), max(low, high)
    result = model.predict(image, conf=max(low, 0.01), verbose=False)[0]
    heads = result.boxes.conf[result.boxes.cls == POS_ID]
    score = float(heads.max()) if len(heads) else 0.0
    decision = "FAIL: alarm" if score >= high else "PASS" if score < low else "REVIEW: send to a person"
    counts = pd.Series(result.boxes.cls.cpu().numpy().astype(int)).map(dict(enumerate(NAMES))).value_counts()
    text = f"### {decision}\nHighest `{POSITIVE}` confidence: {score:.2f}\n\n" + \
        ", ".join(f"{n}: {c}" for n, c in counts.items())
    known = test[test["image"] == Path(image).name]
    if len(known):  # a test-split image, so the label is known
        truth = f"{POSITIVE} present" if known["y_true"].iloc[0] else f"no {POSITIVE}"
        text += f"\n\nTest-split image, ground truth: **{truth}**"
    return result.plot()[..., ::-1], text

## Per-class results and dataset

In [ ]:
def per_class_table():
    rows = []
    for model_name, d in detection.items():
        for cls, m in d["per_class"].items():
            rows.append({"model": model_name, "class": cls, **m})
        rows.append({"model": model_name, "class": "ALL (mean)", "ap50": d["overall"]["map50"],
                     "ap50_95": d["overall"]["map50_95"]})
    return pd.DataFrame(rows).round(3)


def confusion_figure():
    labels = det_cm["labels"]
    fig = go.Figure(go.Heatmap(z=det_cm["matrix"], x=labels[:-1] + ["missed"], y=labels, text=det_cm["matrix"],
                               texttemplate="%{text}", colorscale="Blues"))
    fig.update_layout(xaxis_title="predicted", yaxis_title="true", yaxis_autorange="reversed", height=420,
                      title=f"Finetuned, test split (conf ≥ {det_cm['conf']}, IoU ≥ {det_cm['iou']})")
    return fig


def dataset_table():
    rows = [{"split": s, "class": c, **v} for s, st in dataset_stats.items() for c, v in st["classes"].items()]
    return pd.DataFrame(rows).round(4)

In [ ]:
band = op["band"]
examples_dir = ARTIFACTS / "examples"
examples = pd.read_csv(examples_dir / "examples.csv") if (examples_dir / "examples.csv").exists() else pd.DataFrame()

with gr.Blocks(title="Hard hat compliance") as demo:
    gr.Markdown(f"# Hard hat compliance: when to raise an alarm\nPositive class: **{POSITIVE}** (bare head). "
                f"Chosen single threshold **{op['threshold']:.2f}**, review band **{band['low']:.2f} – {band['high']:.2f}**.")
    with gr.Row():
        low = gr.Slider(0, 1, value=band["low"], step=0.01, label="Pass below (low threshold)")
        high = gr.Slider(0, 1, value=band["high"], step=0.01, label="Fail at or above (high threshold)")
        base_rate = gr.Slider(0.001, 0.5, value=config["base_rate"], step=0.001, label="Base rate (share of cases positive)")

    with gr.Tab("Cost and review"):
        summary = gr.Markdown()
        with gr.Row():
            counts = gr.Dataframe(label="Test split: truth × outcome (image counts)")
            compare = gr.Dataframe(label="Against the alternatives")
        curve = gr.Plot(label="Cost against threshold")

    with gr.Tab("Try an image"):
        with gr.Row():
            image_in = gr.Image(type="filepath", label="Upload")
            image_out = gr.Image(label="Predictions at the low threshold")
        decision = gr.Markdown()
        if len(examples):
            gr.Examples(
                examples=[[str(examples_dir / name)] for name in examples["image"]],
                inputs=[image_in],
                examples_per_page=len(examples),
                label="Or pick a test-split image: two from every truth × outcome cell of the saved band, "
                      "misses included",
            )

    with gr.Tab("Per-class results"):
        gr.Dataframe(per_class_table(), label="Test split, all models")
        gr.Plot(confusion_figure())

    with gr.Tab("Dataset"):
        gr.Dataframe(dataset_table(), label="Instances and relative box area per class and split")

    band_inputs, band_outputs = [low, high, base_rate], [summary, counts, compare, curve]
    # .release fires once when a slider is let go; .change and .input re-fire on their own updates and loop the queue
    for control in band_inputs:
        control.release(band_view, band_inputs, band_outputs, trigger_mode="always_last")
    image_in.change(predict_image, [image_in, low, high], [image_out, decision])
    for control in (low, high):
        control.release(predict_image, [image_in, low, high], [image_out, decision], trigger_mode="always_last")
    demo.load(band_view, band_inputs, band_outputs)

demo.launch()